# Notebook 05: Phase 0 - Precursor Detection

**Date**: 2026-01-30 (Updated)  
**Pipeline**: SPIRAL2 LLRF Anomaly Detection

## Overview

**Purpose**: Detect anomalies in the **PRE-TRIGGER window** to predict fault occurrence before alarms activate.

**Key Concept**: The pre-trigger window (typically 3000 samples ≈ 170ms before fault) contains precursor signatures that may indicate impending failures. By analyzing this window with unsupervised anomaly detection methods, we can potentially provide early warnings.

---

## Ensemble Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    PRECURSOR DETECTION ENSEMBLE                              │
│                  (6 Methods + Majority Voting)                               │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  Input: Pre-trigger features (412 precursor-specific features)              │
│         Derived from 8 LLRF signals × 3000 samples before fault             │
│                                                                              │
├──────────────┬──────────────┬──────────────┬──────────────┬────────────────┤
│              │              │              │              │                │
│  ┌────────┐  │  ┌────────┐  │  ┌────────┐  │  ┌────────┐  │   ┌────────┐   │
│  │Isolation│  │  │  LOF   │  │  │Mahal-  │  │  │  PCA   │  │   │DBSCAN  │   │
│  │ Forest │  │  │        │  │  │anobis  │  │  │Recon.  │  │   │        │   │
│  └────┬───┘  │  └───┬────┘  │  └───┬────┘  │  └───┬────┘  │   └───┬────┘   │
│       │      │      │       │      │       │      │       │       │        │
│  Anomaly     │  Anomaly     │  Distance    │  Recon.     │   Noise        │
│  Score       │  Score       │  Score       │  Error      │   Label        │
│       │      │      │       │      │       │      │       │       │        │
│       v      │      v       │      v       │      v       │       v        │
│  ┌────────────────────────────────────────────────────────────────────┐    │
│  │                    MAJORITY VOTING (τ = 0.5)                       │    │
│  │                                                                    │    │
│  │    Vote = (1/M) Σ predictions  →  Precursor if Vote ≥ 0.5         │    │
│  └────────────────────────────────────────────────────────────────────┘    │
│                                    │                                        │
│                                    v                                        │
│  ┌────────────────────────────────────────────────────────────────────┐    │
│  │  Output: Binary precursor flag + Confidence (vote fraction)        │    │
│  └────────────────────────────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## LSTM Autoencoder Architecture (if sequences available)

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                       LSTM AUTOENCODER                                       │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  Input: Pre-trigger sequence (300 × 8) [downsampled from 3000 × 8]          │
│                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────┐    │
│  │ ENCODER                                                              │    │
│  │  ┌─────────────────────────────────────────────────────────────┐    │    │
│  │  │  LSTM Layer (32 units)                                      │    │    │
│  │  │  Input: (batch, 300, 8) → Output: (batch, 32)              │    │    │
│  │  └─────────────────────────────────────────────────────────────┘    │    │
│  └─────────────────────────────────────────────────────────────────────┘    │
│                              │                                               │
│                              │ Latent z ∈ ℝ³²                               │
│                              │                                               │
│  ┌─────────────────────────────────────────────────────────────────────┐    │
│  │ DECODER                                                              │    │
│  │  ┌─────────────────────────────────────────────────────────────┐    │    │
│  │  │  RepeatVector(300)                                          │    │    │
│  │  │  (batch, 32) → (batch, 300, 32)                            │    │    │
│  │  └─────────────────────────────────────────────────────────────┘    │    │
│  │  ┌─────────────────────────────────────────────────────────────┐    │    │
│  │  │  LSTM Layer (32 units, return_sequences=True)               │    │    │
│  │  │  (batch, 300, 32) → (batch, 300, 32)                       │    │    │
│  │  └─────────────────────────────────────────────────────────────┘    │    │
│  │  ┌─────────────────────────────────────────────────────────────┐    │    │
│  │  │  TimeDistributed(Dense(8))                                  │    │    │
│  │  │  (batch, 300, 32) → (batch, 300, 8)                        │    │    │
│  │  └─────────────────────────────────────────────────────────────┘    │    │
│  └─────────────────────────────────────────────────────────────────────┘    │
│                              │                                               │
│                              v                                               │
│  Output: x̂ (reconstructed sequence)                                        │
│  Anomaly Score: MSE(x, x̂) — higher = more anomalous                        │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## Analysis Strategy
1. **Classical Unsupervised Methods**: Isolation Forest, LOF, DBSCAN, Mahalanobis Distance
2. **Advanced Classical Methods**: PCA Reconstruction Error, HDBSCAN, CUSUM-based Detection
3. **Deep Learning Methods**: LSTM Autoencoder, Variational Autoencoder (VAE)
4. **Ensemble**: Majority voting across all methods

**Input Data**:
- Precursor-specific features from Notebook 03
- Pre-trigger sequences: (n_events, 3000, 8) temporal data
- Multi-label fault targets for validation

**Output**:
- Binary precursor detection flags per event
- Anomaly scores per method
- Ensemble predictions
- Performance metrics (detection rate, false positive rate)

---

## Theory and Background

### Precursor Detection in LLRF Systems

**Physical Motivation**:

In superconducting RF systems, many fault modes exhibit precursor signatures before triggering protection alarms:

1. **Quench Precursors**: Gradual $Q_0$ degradation, increased field emission (detected via GLR rise)
2. **Field Emission Precursors**: Progressive vacuum degradation, rising dark current
3. **Vacuum Fault Precursors**: Slow pressure rise before rapid decompression
4. **Detuning Precursors**: Lorentz force buildup, microphonic resonances

**References**:
- EuXFEL quench detection (thppc072.pdf): Early QL monitoring
- CEBAF LSTM paper: Temporal sequence analysis for fault prediction
- NAS paper: Anomaly scores for early warning systems

### Mathematical Framework

**Precursor Detection Problem**:

Given pre-trigger window $\mathbf{X}_{\text{pre}} \in \mathbb{R}^{T_{\text{pre}} \times d}$ where:
- $T_{\text{pre}} = 3000$ samples (≈170ms)
- $d = 8$ signal channels

**Goal**: Compute anomaly score $s(\mathbf{X}_{\text{pre}})$ such that:

$$s(\mathbf{X}_{\text{pre}}) > \tau \implies \text{fault imminent}$$

where $\tau$ is detection threshold.

**Key Challenge**: No labeled precursor data → unsupervised approach required.

**Evaluation Metric** (post-hoc with fault labels):

$$\text{Detection Rate} = \frac{\text{# faults with detected precursor}}{\text{# total faults}}$$

$$\text{False Positive Rate} = \frac{\text{# normal events flagged as precursor}}{\text{# total normal events}}$$

---

## Notebook Setup

In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Classical ML
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN, OPTICS
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score, roc_curve

# Advanced clustering
try:
    from hdbscan import HDBSCAN
except ImportError:
    print("Warning: hdbscan not installed. Install with: pip install hdbscan")
    HDBSCAN = None

# Deep Learning
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, Model
    TF_AVAILABLE = True
except ImportError:
    print("Warning: TensorFlow not available. Deep learning methods will be skipped.")
    TF_AVAILABLE = False

# Scipy
from scipy.stats import chi2
from scipy.spatial.distance import mahalanobis

# Visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")

2026-01-16 16:59:53.992465: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-16 16:59:54.037140: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-16 16:59:54.350105: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-16 16:59:54.585941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768579194.807896  648367 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768579194.86

✓ All imports successful


In [2]:
# Load features (analysis only - no training)
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('/sps/m4cast/_spiral2_data/_llrf_data/cooked_data')

# Load features
with open(OUTPUT_DIR / 'features_engineered.pkl', 'rb') as f:
    feature_data = pickle.load(f)

X_scaled = feature_data['X_scaled']
X_pca = feature_data['X_pca']
y_binary = feature_data.get('y_binary', None)
y_multilabel = feature_data.get('y_multilabel', None)
fault_column_names = feature_data.get('fault_column_names', [])

print(f"✓ Loaded features: {X_scaled.shape}")

# Load results (if available)
results_file = OUTPUT_DIR / 'step_05_phase0' / 'precursor_detection.pkl'
if results_file.exists():
    with open(results_file, 'rb') as f:
        results = pickle.load(f)
    print(f"✓ Loaded results from pipeline")
else:
    print(f"⚠️  Pipeline results not found: {results_file}")
    print("   Results will be computed in this notebook (may take time)")
    results = None

✓ Loaded features: (4509, 752)
✓ Loaded results from pipeline


## Configuration

Set analysis parameters and hyperparameters for all methods.

In [3]:
# Configuration parameters
CONFIG = {
    'contamination': 0.1,  # Expected fraction of anomalies (10%)
    'n_neighbors': 20,  # For LOF and k-NN
    'pca_components': 20,  # Number of PCA components
    'lstm_latent_dim': 32,  # LSTM autoencoder latent dimension
    'ensemble_threshold': 0.5,  # Majority voting threshold
    'random_seed': 42
}

RANDOM_SEED = CONFIG['random_seed']

# Output paths
OUTPUT_DIR = Path('/sps/m4cast/_spiral2_data/_llrf_data/cooked_data')
OUTPUT_FILE = OUTPUT_DIR / 'step_05_phase0' / 'precursor_detection.pkl'
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print(f"Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nOutput file: {OUTPUT_FILE}")

Configuration:
  contamination: 0.1
  n_neighbors: 20
  pca_components: 20
  lstm_latent_dim: 32
  ensemble_threshold: 0.5
  random_seed: 42

Output file: /sps/m4cast/_spiral2_data/_llrf_data/cooked_data/step_05_phase0/precursor_detection.pkl


In [5]:
# Extract data components already loaded in previous cell
df_features_all = pd.DataFrame(X_scaled)  # Convert to DataFrame if needed
sequences_pretrigger = None  # Will need to load separately if required

print(f"Loaded data:")
print(f"  Scaled features: {X_scaled.shape}")
print(f"  PCA features: {X_pca.shape}")
print(f"  Binary labels: {y_binary.shape if y_binary is not None else 'N/A'}, Faults: {y_binary.sum() if y_binary is not None else 'N/A'}/{len(y_binary) if y_binary is not None else 'N/A'}")
print(f"  Multi-label shape: {y_multilabel.shape if y_multilabel is not None else 'N/A'}")

# Create dummy is_manual if not available
is_manual = np.zeros(len(y_binary) if y_binary is not None else 0, dtype=bool)
print(f"  Manual triggers: {is_manual.sum()}/{len(is_manual)}")

Loaded data:
  Scaled features: (4509, 752)
  PCA features: (4509, 247)
  Binary labels: (4509,), Faults: 1185/4509
  Multi-label shape: (4509, 7)
  Manual triggers: 0/4509


In [4]:
# Extract precursor-specific features from feature names
# Load feature column names
feature_cols = feature_data.get('feature_cols', [f'feature_{i}' for i in range(X_scaled.shape[1])])

precursor_feature_cols = [col for col in feature_cols if any([
    'slope' in col.lower(),
    'accel' in col.lower(),
    'cusum' in col.lower(),
    'deriv' in col.lower(),
    'late' in col.lower(),
    'early' in col.lower(),
    'trend' in col.lower(),
    'change' in col.lower(),
])]

# Get indices of precursor features
precursor_indices = [i for i, col in enumerate(feature_cols) if col in precursor_feature_cols]

if len(precursor_indices) > 0:
    X_precursor = X_scaled[:, precursor_indices]
else:
    # If no precursor features found, use all features
    print("⚠️  No precursor-specific features found, using all features")
    X_precursor = X_scaled
    precursor_feature_cols = feature_cols

print(f"Extracted {len(precursor_feature_cols)} precursor-specific features")
print(f"Precursor feature matrix shape: {X_precursor.shape}")
if len(precursor_feature_cols) > 0:
    print(f"\nExample precursor features:")
    for i, col in enumerate(precursor_feature_cols[:10]):
        print(f"  {i+1}. {col}")

Extracted 412 precursor-specific features
Precursor feature matrix shape: (4509, 412)

Example precursor features:
  1. I modulateur binary_mean
  2. I modulateur binary_min
  3. I modulateur binary_max
  4. I modulateur binary_range
  5. I modulateur binary_median
  6. I modulateur binary_q1
  7. I modulateur binary_skewness
  8. I modulateur binary_kurtosis
  9. I modulateur binary_autocorr_lag1
  10. Q modulateur binary_mean


In [5]:
# Normalize precursor features
scaler_precursor = StandardScaler()
X_precursor_scaled = scaler_precursor.fit_transform(X_precursor)

print(f"Precursor features normalized: {X_precursor_scaled.shape}")
print(f"  Mean: {X_precursor_scaled.mean(axis=0)[:5]} (should be ~0)")
print(f"  Std: {X_precursor_scaled.std(axis=0)[:5]} (should be ~1)")

Precursor features normalized: (4509, 412)
  Mean: [ 1.14247834e-17 -1.89099863e-17  3.15166439e-17  1.89099863e-17
  0.00000000e+00] (should be ~0)
  Std: [1. 1. 1. 1. 1.] (should be ~1)


### Adaptive Parameter Adjustment

**Dataset Size Adaptation**: Adjust algorithm parameters based on actual dataset size to ensure compatibility with both small and large datasets.

**Key Parameters**:
- `n_neighbors` (LOF, k-NN for DBSCAN eps): Must be < n_samples
- `min_samples` (DBSCAN): Core point threshold, scaled with dataset size

**Scaling Rules**:
```python
n_neighbors = min(20, max(2, n_samples - 1))
min_samples = min(5, max(2, n_samples // 5))
```

This ensures:
- Small datasets (n=17): n_neighbors=16, min_samples=3
- Medium datasets (n=100): n_neighbors=20, min_samples=5
- Large datasets (n=500+): n_neighbors=20, min_samples=5


In [6]:
# Adapt CONFIG parameters to dataset size
n_samples = len(X_precursor_scaled)

# Store original values
original_n_neighbors = CONFIG['n_neighbors']

# Adjust n_neighbors for LOF and k-NN (must be < n_samples)
CONFIG['n_neighbors'] = min(CONFIG['n_neighbors'], max(2, n_samples - 1))

# Add adaptive min_samples for DBSCAN (scaled with dataset size)
CONFIG['min_samples_dbscan'] = min(5, max(2, n_samples // 5))

# Print adjustment summary
print(f"Dataset size: {n_samples} samples")
print(f"\nAdaptive parameter adjustment:")
if CONFIG['n_neighbors'] != original_n_neighbors:
    print(f"  n_neighbors: {original_n_neighbors} → {CONFIG['n_neighbors']} (adjusted for small dataset)")
else:
    print(f"  n_neighbors: {CONFIG['n_neighbors']} (no adjustment needed)")
print(f"  min_samples (DBSCAN): {CONFIG['min_samples_dbscan']} (scaled with dataset size)")
print(f"\nUpdated CONFIG: {CONFIG}")

Dataset size: 4509 samples

Adaptive parameter adjustment:
  n_neighbors: 20 (no adjustment needed)
  min_samples (DBSCAN): 5 (scaled with dataset size)

Updated CONFIG: {'contamination': 0.1, 'n_neighbors': 20, 'pca_components': 20, 'lstm_latent_dim': 32, 'ensemble_threshold': 0.5, 'random_seed': 42, 'min_samples_dbscan': 5}


---

## 1. Isolation Forest

### Theory

**Isolation Forest** (Liu et al., 2008) is an unsupervised anomaly detection algorithm based on the principle that anomalies are "few and different," making them easier to isolate.

**Algorithm**:
1. Randomly select a feature and a split value between min/max
2. Recursively partition data until each point is isolated
3. Anomalies require fewer splits → shorter path length

**Anomaly Score**:

$$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$

where:
- $h(x)$ = path length for point $x$
- $E(h(x))$ = average path length over ensemble of trees
- $c(n) = 2H(n-1) - \frac{2(n-1)}{n}$ = normalization factor
- $H(k)$ = harmonic number $\approx \ln(k) + 0.5772$ (Euler's constant)

**Decision Rule**:
- $s \approx 1$: Anomaly (short path)
- $s \approx 0.5$: Normal (average path)
- $s < 0.5$: Very normal (long path)

**Application to LLRF Precursors**:
- Precursor signatures (e.g., CUSUM spikes, slope changes) are rare in pre-trigger windows
- Easier to isolate events with unusual trends compared to stable operation

**Hyperparameters**:
- `n_estimators`: Number of trees (default 100)
- `contamination`: Expected proportion of anomalies (set to 0.1 = 10%)
- `max_samples`: Subsample size (default 256)

**References**:
- Liu et al. (2008), "Isolation Forest", IEEE ICDM
- EuXFEL anomaly detection: Similar isolation-based approach for quench precursors


In [7]:
# Train Isolation Forest on precursor features
print("Training Isolation Forest...")

iforest = IsolationForest(
    n_estimators=100,
    contamination=CONFIG['contamination'],
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=0
)

# Fit on precursor features
iforest.fit(X_precursor_scaled)

# Predict: -1 = anomaly, 1 = normal
iforest_predictions = iforest.predict(X_precursor_scaled)
iforest_anomaly_flags = (iforest_predictions == -1).astype(int)

# Anomaly scores (lower = more anomalous)
iforest_scores = -iforest.score_samples(X_precursor_scaled)  # Negate so higher = more anomalous

print(f"✓ Isolation Forest trained")
print(f"  Detected anomalies: {iforest_anomaly_flags.sum()}/{len(iforest_anomaly_flags)}")
print(f"  Anomaly score range: [{iforest_scores.min():.3f}, {iforest_scores.max():.3f}]")

Training Isolation Forest...
✓ Isolation Forest trained
  Detected anomalies: 451/4509
  Anomaly score range: [0.326, 0.656]


In [8]:
# Evaluate against fault labels
iforest_detection_rate = (iforest_anomaly_flags[y_binary == 1]).mean()
iforest_fpr = (iforest_anomaly_flags[y_binary == 0]).mean()

print(f"Isolation Forest Performance:")
print(f"  Detection Rate (recall on faults): {iforest_detection_rate:.2%}")
print(f"  False Positive Rate (on normal events): {iforest_fpr:.2%}")

Isolation Forest Performance:
  Detection Rate (recall on faults): 5.49%
  False Positive Rate (on normal events): 11.61%


---

## 2. Local Outlier Factor (LOF)

### Theory

**LOF** (Breunig et al., 2000) measures the local density deviation of a point compared to its neighbors. Points in low-density regions are flagged as anomalies.

**Algorithm**:

1. **K-distance**: Distance to $k$-th nearest neighbor
   $$k\text{-dist}(x) = d(x, N_k(x))$$

2. **Reachability Distance**:
   $$\text{reach-dist}_k(x, y) = \max\{k\text{-dist}(y), d(x, y)\}$$
   (Smooths distance to avoid statistical fluctuations)

3. **Local Reachability Density (LRD)**:
   $$\text{LRD}_k(x) = \left( \frac{1}{|N_k(x)|} \sum_{y \in N_k(x)} \text{reach-dist}_k(x, y) \right)^{-1}$$
   Higher LRD → denser neighborhood

4. **Local Outlier Factor**:
   $$\text{LOF}_k(x) = \frac{1}{|N_k(x)|} \sum_{y \in N_k(x)} \frac{\text{LRD}_k(y)}{\text{LRD}_k(x)}$$

**Interpretation**:
- $\text{LOF} \approx 1$: Similar density to neighbors (normal)
- $\text{LOF} \gg 1$: Much lower density than neighbors (anomaly)
- $\text{LOF} < 1$: Higher density than neighbors (very normal)

**Application to LLRF**:
- Precursor events are isolated in feature space (e.g., high CUSUM values)
- LOF detects these low-density regions effectively

**Hyperparameters**:
- `n_neighbors`: Number of neighbors for density estimation (default 20)
- `contamination`: Expected anomaly fraction

**References**:
- Breunig et al. (2000), "LOF: Identifying Density-Based Local Outliers", ACM SIGMOD
- Used in EuXFEL NAS paper for outlier detection in cavity signals


In [9]:
# Train LOF on precursor features
print("Training Local Outlier Factor...")

lof = LocalOutlierFactor(
    n_neighbors=CONFIG['n_neighbors'],
    contamination=CONFIG['contamination'],
    novelty=False,  # Transductive mode (fit_predict on same data)
    n_jobs=-1
)

# Fit and predict in one step
lof_predictions = lof.fit_predict(X_precursor_scaled)
lof_anomaly_flags = (lof_predictions == -1).astype(int)

# Negative outlier factor (more negative = more anomalous)
lof_scores = -lof.negative_outlier_factor_  # Negate so higher = more anomalous

print(f"✓ LOF trained")
print(f"  Detected anomalies: {lof_anomaly_flags.sum()}/{len(lof_anomaly_flags)}")
print(f"  LOF score range: [{lof_scores.min():.3f}, {lof_scores.max():.3f}]")

Training Local Outlier Factor...
✓ LOF trained
  Detected anomalies: 451/4509
  LOF score range: [0.974, 13.603]


In [12]:
# Evaluate against fault labels
lof_detection_rate = (lof_anomaly_flags[y_binary == 1]).mean()
lof_fpr = (lof_anomaly_flags[y_binary == 0]).mean()

print(f"LOF Performance:")
print(f"  Detection Rate: {lof_detection_rate:.2%}")
print(f"  False Positive Rate: {lof_fpr:.2%}")

LOF Performance:
  Detection Rate: 10.80%
  False Positive Rate: 9.72%


---

## 3. Mahalanobis Distance

### Theory

**Mahalanobis Distance** measures how many standard deviations a point is from the mean of a distribution, accounting for covariance.

**Formula**:

$$D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}$$

where:
- $\mu$ = mean vector
- $\Sigma$ = covariance matrix
- $\Sigma^{-1}$ = inverse covariance (precision matrix)

**Statistical Test**:

Under multivariate normal assumption, $D_M^2$ follows $\chi^2$ distribution with $p$ degrees of freedom.

**Threshold for Anomaly**:

$$D_M^2(x) > \chi^2_{p, 1-\alpha}$$

where $\alpha$ is significance level (e.g., 0.05 for 95% confidence).

**Advantages**:
- Accounts for correlations between features
- Scale-invariant
- Theoretically grounded (statistical test)

**Application to LLRF**:
- Features are correlated (e.g., Ucav and PhaseCav)
- Mahalanobis distance naturally handles this correlation
- Precursors deviate from normal operation's multivariate distribution

**References**:
- Mahalanobis (1936), "On the Generalized Distance in Statistics"
- De Maesschalck et al. (2000), "The Mahalanobis distance", Chemometrics


In [10]:
# Compute Mahalanobis distance
print("Computing Mahalanobis distances...")

# Estimate mean and covariance from data
mu = X_precursor_scaled.mean(axis=0)
cov = np.cov(X_precursor_scaled, rowvar=False)

# Add regularization to ensure invertibility
cov_reg = cov + np.eye(cov.shape[0]) * 1e-6

# Invert covariance matrix
try:
    cov_inv = np.linalg.inv(cov_reg)
except np.linalg.LinAlgError:
    print("Warning: Covariance matrix singular, using pseudoinverse")
    cov_inv = np.linalg.pinv(cov_reg)

# Compute Mahalanobis distance for each point
mahal_distances = np.array([
    mahalanobis(x, mu, cov_inv) for x in X_precursor_scaled
])

# Chi-squared threshold for anomaly detection
p = X_precursor_scaled.shape[1]  # Number of features
alpha = 0.05  # Significance level
threshold_chisq = chi2.ppf(1 - alpha, df=p)
threshold_mahal = np.sqrt(threshold_chisq)

mahal_anomaly_flags = (mahal_distances > threshold_mahal).astype(int)
mahal_scores = mahal_distances  # Use distance as score

print(f"✓ Mahalanobis distances computed")
print(f"  Threshold (95% confidence): {threshold_mahal:.3f}")
print(f"  Detected anomalies: {mahal_anomaly_flags.sum()}/{len(mahal_anomaly_flags)}")
print(f"  Distance range: [{mahal_distances.min():.3f}, {mahal_distances.max():.3f}]")

Computing Mahalanobis distances...
✓ Mahalanobis distances computed
  Threshold (95% confidence): 21.455
  Detected anomalies: 779/4509
  Distance range: [5.120, 67.134]


In [11]:
# Evaluate
mahal_detection_rate = (mahal_anomaly_flags[y_binary == 1]).mean()
mahal_fpr = (mahal_anomaly_flags[y_binary == 0]).mean()

print(f"Mahalanobis Distance Performance:")
print(f"  Detection Rate: {mahal_detection_rate:.2%}")
print(f"  False Positive Rate: {mahal_fpr:.2%}")

Mahalanobis Distance Performance:
  Detection Rate: 15.53%
  False Positive Rate: 17.90%


---

## 4. PCA Reconstruction Error

### Theory

**PCA-based Anomaly Detection** assumes normal data lies in a low-dimensional subspace. Anomalies have high reconstruction error when projected onto this subspace.

**Algorithm**:

1. **Compute PCA**: Find principal components $\mathbf{W} \in \mathbb{R}^{p \times k}$ (keep $k$ components)

2. **Project to latent space**:
   $$z = \mathbf{W}^T x$$

3. **Reconstruct**:
   $$\hat{x} = \mathbf{W} z = \mathbf{W} \mathbf{W}^T x$$

4. **Reconstruction Error**:
   $$E(x) = \|x - \hat{x}\|^2 = \|x - \mathbf{W}\mathbf{W}^T x\|^2$$

**Anomaly Score**:

$$s(x) = \sqrt{E(x)}$$

Higher reconstruction error → anomaly.

**Statistical Threshold** (Hotelling's T²):

$$T^2 = z^T \Lambda^{-1} z \sim \chi^2_k$$

where $\Lambda$ is diagonal matrix of eigenvalues.

**Application to LLRF**:
- Normal operation occupies low-dimensional manifold
- Precursors deviate from this manifold (e.g., unusual CUSUM patterns)

**Hyperparameters**:
- `n_components`: Number of components to retain (variance-based or fixed)

**References**:
- Shyu et al. (2003), "A Novel Anomaly Detection Scheme Based on PCA", IEEE
- Jackson & Mudholkar (1979), "Control procedures for residuals", Technometrics


In [12]:
# PCA reconstruction error
print("Computing PCA reconstruction error...")

# Fit PCA on precursor features
pca_precursor = PCA(n_components=CONFIG['pca_components'], random_state=RANDOM_SEED)
pca_precursor.fit(X_precursor_scaled)

n_components_kept = pca_precursor.n_components_
print(f"  PCA retained {n_components_kept} components (explaining {pca_precursor.explained_variance_ratio_.sum():.1%} variance)")

# Project and reconstruct
X_projected = pca_precursor.transform(X_precursor_scaled)
X_reconstructed = pca_precursor.inverse_transform(X_projected)

# Reconstruction error
pca_recon_errors = np.linalg.norm(X_precursor_scaled - X_reconstructed, axis=1)

# Threshold: Mean + 3*std (heuristic)
threshold_pca = pca_recon_errors.mean() + 3 * pca_recon_errors.std()
pca_anomaly_flags = (pca_recon_errors > threshold_pca).astype(int)
pca_scores = pca_recon_errors

print(f"✓ PCA reconstruction complete")
print(f"  Threshold: {threshold_pca:.3f}")
print(f"  Detected anomalies: {pca_anomaly_flags.sum()}/{len(pca_anomaly_flags)}")
print(f"  Reconstruction error range: [{pca_recon_errors.min():.3f}, {pca_recon_errors.max():.3f}]")

Computing PCA reconstruction error...
  PCA retained 20 components (explaining 40.5% variance)
✓ PCA reconstruction complete
  Threshold: 40.747
  Detected anomalies: 102/4509
  Reconstruction error range: [3.586, 112.612]


In [13]:
# Evaluate
pca_detection_rate = (pca_anomaly_flags[y_binary == 1]).mean()
pca_fpr = (pca_anomaly_flags[y_binary == 0]).mean()

print(f"PCA Reconstruction Error Performance:")
print(f"  Detection Rate: {pca_detection_rate:.2%}")
print(f"  False Positive Rate: {pca_fpr:.2%}")

PCA Reconstruction Error Performance:
  Detection Rate: 2.28%
  False Positive Rate: 2.26%


---

## 5. DBSCAN Clustering

### Theory

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) identifies clusters based on density and labels low-density points as anomalies.

**Definitions**:

1. **ε-neighborhood**: $N_\varepsilon(x) = \{y \in D : d(x, y) \leq \varepsilon\}$

2. **Core point**: $|N_\varepsilon(x)| \geq \text{MinPts}$ (dense region)

3. **Border point**: Not core, but in neighborhood of core point

4. **Noise point**: Neither core nor border → **ANOMALY**

**Algorithm**:
1. For each unvisited point $x$:
   - If $x$ is core point: Start new cluster, expand via density reachability
   - Else: Mark as noise (potential anomaly)

**Anomaly Detection**:
- Points labeled as noise (cluster ID = -1) are anomalies

**Hyperparameters**:
- `eps`: Maximum distance for neighborhood (ε)
- `min_samples`: Minimum points for core status (MinPts)

**Application to LLRF**:
- Precursor events form sparse regions in feature space
- DBSCAN's noise points correspond to rare precursor signatures

**References**:
- Ester et al. (1996), "A Density-Based Algorithm for Discovering Clusters", KDD
- EuXFEL: Clustering-based outlier detection in cavity performance data


In [14]:
# DBSCAN clustering
print("Running DBSCAN...")

# Heuristic: eps as 90th percentile of k-nearest neighbor distances
from sklearn.neighbors import NearestNeighbors
k = CONFIG['n_neighbors']
nbrs = NearestNeighbors(n_neighbors=k).fit(X_precursor_scaled)
distances, indices = nbrs.kneighbors(X_precursor_scaled)
eps_estimate = np.percentile(distances[:, -1], 90)

dbscan = DBSCAN(
    eps=eps_estimate,
    min_samples=CONFIG['min_samples_dbscan'],
    n_jobs=-1
)

dbscan_labels = dbscan.fit_predict(X_precursor_scaled)

# Noise points (label -1) are anomalies
dbscan_anomaly_flags = (dbscan_labels == -1).astype(int)

# For scoring, use distance to nearest cluster center as proxy
# (Higher distance = more anomalous)
unique_labels = np.unique(dbscan_labels[dbscan_labels != -1])
if len(unique_labels) > 0:
    cluster_centers = np.array([
        X_precursor_scaled[dbscan_labels == label].mean(axis=0)
        for label in unique_labels
    ])

    dbscan_scores = np.zeros(len(X_precursor_scaled))
    for i, x in enumerate(X_precursor_scaled):
        if dbscan_labels[i] == -1:
            # Noise: distance to nearest cluster center
            dists = np.linalg.norm(cluster_centers - x, axis=1)
            dbscan_scores[i] = dists.min()
        else:
            # Clustered: distance to own cluster center
            center = cluster_centers[unique_labels == dbscan_labels[i]][0]
            dbscan_scores[i] = np.linalg.norm(x - center)
else:
    dbscan_scores = np.ones(len(X_precursor_scaled))

print(f"✓ DBSCAN complete")
print(f"  eps estimate: {eps_estimate:.3f}")
print(f"  min_samples: {CONFIG['min_samples_dbscan']}")
print(f"  Number of clusters: {len(unique_labels)}")
print(f"  Noise points (anomalies): {dbscan_anomaly_flags.sum()}/{len(dbscan_anomaly_flags)}")

Running DBSCAN...
✓ DBSCAN complete
  eps estimate: 21.966
  min_samples: 5
  Number of clusters: 2
  Noise points (anomalies): 270/4509


In [15]:
# Evaluate
dbscan_detection_rate = (dbscan_anomaly_flags[y_binary == 1]).mean()
dbscan_fpr = (dbscan_anomaly_flags[y_binary == 0]).mean()

print(f"DBSCAN Performance:")
print(f"  Detection Rate: {dbscan_detection_rate:.2%}")
print(f"  False Positive Rate: {dbscan_fpr:.2%}")

DBSCAN Performance:
  Detection Rate: 4.98%
  False Positive Rate: 6.35%


---

## 6. LSTM Autoencoder (Deep Learning)

### Theory

**LSTM Autoencoder** learns to compress and reconstruct temporal sequences. Normal sequences are well-reconstructed; anomalous sequences have high reconstruction error.

**Architecture**:

**Encoder**:
$$h_t^{(enc)} = \text{LSTM}(x_t, h_{t-1}^{(enc)})$$
$$z = h_T^{(enc)} \quad \text{(latent representation)}$$

**Decoder**:
$$h_t^{(dec)} = \text{LSTM}(z, h_{t-1}^{(dec)})$$
$$\hat{x}_t = \text{Dense}(h_t^{(dec)})$$

**Loss Function** (Mean Squared Error):
$$\mathcal{L} = \frac{1}{T \cdot d} \sum_{t=1}^{T} \sum_{j=1}^{d} (x_{t,j} - \hat{x}_{t,j})^2$$

**Anomaly Score**:
$$s(x) = \sqrt{\mathcal{L}(x)}$$

**Training**:
- Train only on normal operation data (if available)
- Or train on all data and use high reconstruction error as anomaly indicator

**Application to LLRF**:
- Pre-trigger sequences: (3000 samples, 8 channels)
- LSTM captures temporal dependencies (trends, oscillations)
- Precursors exhibit unusual temporal patterns → high reconstruction error

**Hyperparameters**:
- `latent_dim`: Bottleneck size (compression)
- `lstm_units`: Number of LSTM units
- `epochs`, `batch_size`: Training parameters

**References**:
- Malhotra et al. (2016), "LSTM-based Encoder-Decoder for Multi-sensor Anomaly Detection", ICML
- CEBAF LSTM paper: Similar approach for LLRF fault prediction
- Variational autoencoders for time series anomaly detection (Su et al., 2019)


### Load Pre-Trigger Sequences

Before training the LSTM autoencoder, we need to load the actual pre-trigger sequence data from the batch files.

In [ ]:
# Check for pre-computed sequences in features file, or skip LSTM
print("Checking for pre-trigger sequence data...")

sequences_pretrigger = None

# Option 1: Check if sequences are stored in features file
if 'sequences_pretrigger' in feature_data:
    sequences_pretrigger = feature_data['sequences_pretrigger']
    print(f"✓ Found pre-trigger sequences in features file: {sequences_pretrigger.shape}")

# Option 2: Check for memory-mapped sequences file
elif (OUTPUT_DIR / 'sequences_pretrigger.npy').exists():
    sequences_pretrigger = np.load(OUTPUT_DIR / 'sequences_pretrigger.npy', mmap_mode='r')
    print(f"✓ Found memory-mapped sequences: {sequences_pretrigger.shape}")

# Option 3: Try to load from batches with memory limit
else:
    print("⚠️  Pre-trigger sequences not available in features file.")
    print("   Loading from batches would require too much memory (~3GB+).")
    print("   LSTM Autoencoder will be SKIPPED.")
    print("")
    print("   To enable LSTM analysis, run the pipeline with store_signals=True")
    print("   or create a separate sequence extraction script.")
    sequences_pretrigger = None

# Summary
if sequences_pretrigger is not None:
    print(f"\nSequence data available:")
    print(f"  Shape: {sequences_pretrigger.shape}")
    print(f"  Memory: {sequences_pretrigger.nbytes / (1024**2):.1f} MB")
else:
    print("\n⚠️  LSTM Autoencoder will use FEATURE-BASED approach instead of sequences")
    print("   This is less powerful but avoids memory issues.")

In [ ]:
# LSTM Autoencoder (only if TensorFlow available)
if TF_AVAILABLE and sequences_pretrigger is not None and len(sequences_pretrigger) > 0:
    print("Building LSTM Autoencoder...")

    # Check if sequence count matches feature count
    if len(sequences_pretrigger) != len(X_scaled):
        print(f"⚠️  Warning: Using {len(sequences_pretrigger)} sequences (doesn't match {len(X_scaled)} features)")
        print(f"   LSTM results will only apply to subset of events with valid sequences")
        # Create mapping for valid sequence indices
        n_total = len(X_scaled)
        lstm_anomaly_flags = np.zeros(n_total, dtype=int)
        lstm_scores = np.zeros(n_total)
    
    # Prepare sequences: (n_events, 3000, 8) → downsample to (n_events, 300, 8) for faster training
    downsample_factor = 10
    sequences_downsampled = sequences_pretrigger[:, ::downsample_factor, :]
    n_events, seq_len, n_features = sequences_downsampled.shape

    print(f"  Downsampled sequences: {sequences_downsampled.shape}")

    # Normalize sequences (per-channel)
    seq_mean = sequences_downsampled.mean(axis=(0, 1), keepdims=True)
    seq_std = sequences_downsampled.std(axis=(0, 1), keepdims=True) + 1e-8
    sequences_normalized = (sequences_downsampled - seq_mean) / seq_std

    # Split: Train on 80% (ideally normal data, but we use all for unsupervised)
    n_train = int(0.8 * n_events)
    X_train_seq = sequences_normalized[:n_train]
    X_test_seq = sequences_normalized[n_train:]

    # Build LSTM Autoencoder
    latent_dim = CONFIG['lstm_latent_dim']

    # Encoder
    encoder_inputs = keras.Input(shape=(seq_len, n_features))
    encoder_lstm = layers.LSTM(latent_dim, return_state=False)(encoder_inputs)

    # Decoder
    decoder_lstm = layers.RepeatVector(seq_len)(encoder_lstm)
    decoder_lstm = layers.LSTM(latent_dim, return_sequences=True)(decoder_lstm)
    decoder_outputs = layers.TimeDistributed(layers.Dense(n_features))(decoder_lstm)

    # Autoencoder model
    lstm_autoencoder = keras.Model(encoder_inputs, decoder_outputs, name='lstm_autoencoder')
    lstm_autoencoder.compile(optimizer='adam', loss='mse')

    print(lstm_autoencoder.summary())

    # Train
    print("Training LSTM Autoencoder (this may take a few minutes)...")
    history = lstm_autoencoder.fit(
        X_train_seq, X_train_seq,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    )

    print(f"✓ LSTM Autoencoder trained (final loss: {history.history['loss'][-1]:.4f})")

    # Predict and compute reconstruction error
    X_reconstructed = lstm_autoencoder.predict(sequences_normalized, verbose=0)
    lstm_recon_errors_seq = np.mean((sequences_normalized - X_reconstructed)**2, axis=(1, 2))

    # Threshold: Mean + 3*std
    threshold_lstm = lstm_recon_errors_seq.mean() + 3 * lstm_recon_errors_seq.std()
    
    if len(sequences_pretrigger) == len(X_scaled):
        # Direct assignment
        lstm_anomaly_flags = (lstm_recon_errors_seq > threshold_lstm).astype(int)
        lstm_scores = lstm_recon_errors_seq
    else:
        # Need to map back to original indices (only process first N events for now)
        lstm_scores[:len(lstm_recon_errors_seq)] = lstm_recon_errors_seq
        lstm_anomaly_flags[:len(lstm_recon_errors_seq)] = (lstm_recon_errors_seq > threshold_lstm).astype(int)

    print(f"  Threshold: {threshold_lstm:.6f}")
    print(f"  Detected anomalies: {lstm_anomaly_flags.sum()}/{len(lstm_anomaly_flags)}")
    print(f"  Reconstruction error range: [{lstm_recon_errors_seq.min():.6f}, {lstm_recon_errors_seq.max():.6f}]")

    # Evaluate (only on events with sequences if mismatch)
    if len(sequences_pretrigger) == len(X_scaled):
        lstm_detection_rate = (lstm_anomaly_flags[y_binary == 1]).mean()
        lstm_fpr = (lstm_anomaly_flags[y_binary == 0]).mean()
    else:
        # Evaluate only on events with sequences
        y_binary_subset = y_binary[:len(sequences_pretrigger)]
        lstm_anomaly_flags_subset = lstm_anomaly_flags[:len(sequences_pretrigger)]
        lstm_detection_rate = (lstm_anomaly_flags_subset[y_binary_subset == 1]).mean()
        lstm_fpr = (lstm_anomaly_flags_subset[y_binary_subset == 0]).mean()

    print(f"\nLSTM Autoencoder Performance:")
    print(f"  Detection Rate: {lstm_detection_rate:.2%}")
    print(f"  False Positive Rate: {lstm_fpr:.2%}")
    
elif not TF_AVAILABLE:
    print("TensorFlow not available. Skipping LSTM Autoencoder.")
    lstm_anomaly_flags = np.zeros(len(y_binary), dtype=int)
    lstm_scores = np.zeros(len(y_binary))
    lstm_detection_rate = 0.0
    lstm_fpr = 0.0
else:
    print("No pre-trigger sequences available. Skipping LSTM Autoencoder.")
    lstm_anomaly_flags = np.zeros(len(y_binary), dtype=int)
    lstm_scores = np.zeros(len(y_binary))
    lstm_detection_rate = 0.0
    lstm_fpr = 0.0

---

## 7. Ensemble Method

### Theory

**Ensemble Learning** combines predictions from multiple models to improve robustness and accuracy.

**Majority Voting**:

$$\hat{y}_{\text{ensemble}} = \mathbb{1}\left[\frac{1}{M} \sum_{m=1}^{M} \hat{y}_m > \tau\right]$$

where:
- $M$ = number of methods
- $\hat{y}_m \in \{0, 1\}$ = prediction from method $m$
- $\tau$ = voting threshold (default 0.5 for majority)

**Advantages**:
- Reduces variance (different methods have different biases)
- More robust to method-specific failures
- Interpretable (agreement = confidence)

**Application to LLRF**:
- Combine Isolation Forest, LOF, Mahalanobis, PCA, DBSCAN, LSTM
- If ≥ 50% methods flag precursor → ensemble flags precursor
- High agreement → high confidence in detection

**Confidence Metric**:
$$\text{Confidence} = \frac{1}{M} \sum_{m=1}^{M} \hat{y}_m$$

Values near 0 or 1 indicate high confidence; values near 0.5 indicate uncertainty.


In [ ]:
# Ensemble: Majority voting
print("Creating ensemble predictions...")

# Collect all predictions
method_names = ['IsolationForest', 'LOF', 'Mahalanobis', 'PCA', 'DBSCAN']
predictions_matrix = np.column_stack([
    iforest_anomaly_flags,
    lof_anomaly_flags,
    mahal_anomaly_flags,
    pca_anomaly_flags,
    dbscan_anomaly_flags,
])

if TF_AVAILABLE:
    method_names.append('LSTM_AE')
    predictions_matrix = np.column_stack([predictions_matrix, lstm_anomaly_flags])

# Majority voting
vote_fractions = predictions_matrix.mean(axis=1)
ensemble_anomaly_flags = (vote_fractions >= CONFIG['ensemble_threshold']).astype(int)

print(f"✓ Ensemble created from {len(method_names)} methods")
print(f"  Voting threshold: {CONFIG['ensemble_threshold']}")
print(f"  Detected anomalies: {ensemble_anomaly_flags.sum()}/{len(ensemble_anomaly_flags)}")

# Evaluate
ensemble_detection_rate = (ensemble_anomaly_flags[y_binary == 1]).mean()
ensemble_fpr = (ensemble_anomaly_flags[y_binary == 0]).mean()

print(f"\nEnsemble Performance:")
print(f"  Detection Rate: {ensemble_detection_rate:.2%}")
print(f"  False Positive Rate: {ensemble_fpr:.2%}")

---

## 8. Performance Summary

Compile all results into a summary table.

In [ ]:
# Create performance summary table
from sklearn.metrics import precision_recall_fscore_support

# Compile results
results_summary = pd.DataFrame({
    'Method': ['IsolationForest', 'LOF', 'Mahalanobis', 'PCA', 'DBSCAN'],
    'Detection_Rate': [iforest_detection_rate, lof_detection_rate, mahal_detection_rate, pca_detection_rate, dbscan_detection_rate],
    'False_Positive_Rate': [iforest_fpr, lof_fpr, mahal_fpr, pca_fpr, dbscan_fpr]
})

# Compute F1 scores
for idx, method in enumerate(results_summary['Method']):
    if method == 'IsolationForest':
        preds = iforest_anomaly_flags
    elif method == 'LOF':
        preds = lof_anomaly_flags
    elif method == 'Mahalanobis':
        preds = mahal_anomaly_flags
    elif method == 'PCA':
        preds = pca_anomaly_flags
    elif method == 'DBSCAN':
        preds = dbscan_anomaly_flags
    
    if y_binary is not None:
        precision, recall, f1, _ = precision_recall_fscore_support(y_binary, preds, average='binary', zero_division=0)
        results_summary.loc[idx, 'F1_Score'] = f1
    else:
        results_summary.loc[idx, 'F1_Score'] = 0.0

# Add LSTM if available
if TF_AVAILABLE and 'lstm_detection_rate' in globals():
    lstm_row = pd.DataFrame({
        'Method': ['LSTM_AE'],
        'Detection_Rate': [lstm_detection_rate],
        'False_Positive_Rate': [lstm_fpr]
    })
    if y_binary is not None:
        precision, recall, f1, _ = precision_recall_fscore_support(y_binary, lstm_anomaly_flags, average='binary', zero_division=0)
        lstm_row['F1_Score'] = f1
    else:
        lstm_row['F1_Score'] = 0.0
    results_summary = pd.concat([results_summary, lstm_row], ignore_index=True)

# Add ensemble
ensemble_row = pd.DataFrame({
    'Method': ['Ensemble'],
    'Detection_Rate': [ensemble_detection_rate],
    'False_Positive_Rate': [ensemble_fpr]
})
if y_binary is not None:
    precision, recall, f1, _ = precision_recall_fscore_support(y_binary, ensemble_anomaly_flags, average='binary', zero_division=0)
    ensemble_row['F1_Score'] = f1
else:
    ensemble_row['F1_Score'] = 0.0

results_summary = pd.concat([results_summary, ensemble_row], ignore_index=True)

print("Performance Summary:")
print(results_summary.to_string(index=False))

# Visualize performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Detection Rate
axes[0].barh(results_summary['Method'], results_summary['Detection_Rate'], color='skyblue')
axes[0].set_xlabel('Detection Rate (Recall on Faults)', fontsize=12)
axes[0].set_title('Precursor Detection Rate', fontsize=14, fontweight='bold')
axes[0].set_xlim([0, 1])
axes[0].grid(axis='x', alpha=0.3)

# False Positive Rate
axes[1].barh(results_summary['Method'], results_summary['False_Positive_Rate'], color='salmon')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_title('False Positive Rate', fontsize=14, fontweight='bold')
axes[1].set_xlim([0, max(0.5, results_summary['False_Positive_Rate'].max() * 1.1)])
axes[1].grid(axis='x', alpha=0.3)

# F1-Score
axes[2].barh(results_summary['Method'], results_summary['F1_Score'], color='lightgreen')
axes[2].set_xlabel('F1-Score', fontsize=12)
axes[2].set_title('F1-Score', fontsize=14, fontweight='bold')
axes[2].set_xlim([0, 1])
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()

# Create outputs directory if needed
outputs_dir = Path('../outputs')
outputs_dir.mkdir(exist_ok=True)
plt.savefig(outputs_dir / 'phase0_performance_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Performance visualization saved to {outputs_dir / 'phase0_performance_summary.png'}")

## Confusion Matrices

Evaluate precursor detection as binary classification task (precursor detected vs. not detected).


In [ ]:
# Plot confusion matrices for all methods
if y_binary is not None:
    n_methods = len(method_names) + 1  # +1 for ensemble
    n_cols = 3
    n_rows = int(np.ceil(n_methods / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten()

    all_predictions = [
        iforest_anomaly_flags,
        lof_anomaly_flags,
        mahal_anomaly_flags,
        pca_anomaly_flags,
        dbscan_anomaly_flags,
    ]
    if TF_AVAILABLE and 'lstm_anomaly_flags' in globals():
        all_predictions.append(lstm_anomaly_flags)
    all_predictions.append(ensemble_anomaly_flags)

    for idx, (method, preds) in enumerate(zip(method_names + ['Ensemble'], all_predictions)):
        cm = confusion_matrix(y_binary, preds)

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=['No Fault', 'Fault'],
                    yticklabels=['No Fault', 'Fault'],
                    ax=axes[idx])
        axes[idx].set_title(f'{method}', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('True Label')
        axes[idx].set_xlabel('Predicted Precursor')

    # Hide unused subplots
    for idx in range(n_methods, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    
    outputs_dir = Path('../outputs')
    outputs_dir.mkdir(exist_ok=True)
    plt.savefig(outputs_dir / 'phase0_confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"✓ Confusion matrices saved to {outputs_dir / 'phase0_confusion_matrices.png'}")
else:
    print("⚠️  No labels available for confusion matrix")

## Anomaly Score Distributions

Visualize score distributions for fault vs. normal events.


In [ ]:
# Plot score distributions
if y_binary is not None:
    all_scores = [
        ('IsolationForest', iforest_scores),
        ('LOF', lof_scores),
        ('Mahalanobis', mahal_scores),
        ('PCA', pca_scores),
        ('DBSCAN', dbscan_scores),
    ]
    if TF_AVAILABLE and 'lstm_scores' in globals():
        all_scores.append(('LSTM_AE', lstm_scores))

    n_methods_score = len(all_scores)
    n_cols = 2
    n_rows = int(np.ceil(n_methods_score / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = axes.flatten()

    for idx, (method, scores) in enumerate(all_scores):
        # Separate scores by true label
        scores_normal = scores[y_binary == 0]
        scores_fault = scores[y_binary == 1]

        axes[idx].hist(scores_normal, bins=30, alpha=0.6, label='Normal', color='blue')
        axes[idx].hist(scores_fault, bins=30, alpha=0.6, label='Fault', color='red')
        axes[idx].set_xlabel('Anomaly Score', fontsize=11)
        axes[idx].set_ylabel('Frequency', fontsize=11)
        axes[idx].set_title(f'{method} - Score Distribution', fontsize=12, fontweight='bold')
        axes[idx].legend()
        axes[idx].grid(alpha=0.3)

    # Hide unused subplots
    for idx in range(n_methods_score, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    
    outputs_dir = Path('../outputs')
    outputs_dir.mkdir(exist_ok=True)
    plt.savefig(outputs_dir / 'phase0_score_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"✓ Score distributions saved to {outputs_dir / 'phase0_score_distributions.png'}")
else:
    print("⚠️  No labels available for score distribution")

## Ensemble Agreement Analysis

Analyze how often methods agree on precursor detection.


In [ ]:
# Method agreement heatmap
# Compute pairwise agreement (Jaccard similarity)
n_methods_total = len(method_names)
agreement_matrix = np.zeros((n_methods_total, n_methods_total))

for i in range(n_methods_total):
    for j in range(n_methods_total):
        pred_i = predictions_matrix[:, i]
        pred_j = predictions_matrix[:, j]

        # Jaccard: intersection / union
        intersection = (pred_i & pred_j).sum()
        union = (pred_i | pred_j).sum()
        agreement_matrix[i, j] = intersection / (union + 1e-10)

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(agreement_matrix, annot=True, fmt='.2f', cmap='YlGnBu',
            xticklabels=method_names, yticklabels=method_names,
            vmin=0, vmax=1, cbar_kws={'label': 'Jaccard Similarity'})
plt.title('Method Agreement Matrix (Precursor Detection)', fontsize=14, fontweight='bold')
plt.tight_layout()

outputs_dir = Path('../outputs')
outputs_dir.mkdir(exist_ok=True)
plt.savefig(outputs_dir / 'phase0_method_agreement.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Agreement matrix saved to {outputs_dir / 'phase0_method_agreement.png'}")

In [ ]:
# Vote distribution
plt.figure(figsize=(10, 6))
plt.hist(vote_fractions, bins=np.linspace(0, 1, 12), edgecolor='black', alpha=0.7, color='teal')
plt.axvline(CONFIG['ensemble_threshold'], color='red', linestyle='--', linewidth=2, label=f'Threshold = {CONFIG["ensemble_threshold"]}')
plt.xlabel('Fraction of Methods Voting Anomaly', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.title('Ensemble Vote Distribution', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()

outputs_dir = Path('../outputs')
outputs_dir.mkdir(exist_ok=True)
plt.savefig(outputs_dir / 'phase0_vote_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Vote distribution saved to {outputs_dir / 'phase0_vote_distribution.png'}")

---

## 9. Save Results

In [ ]:
# Prepare output dictionary
phase0_results = {
    # Predictions
    'precursor_detected': ensemble_anomaly_flags,
    'vote_fractions': vote_fractions,

    # Individual method results
    'method_predictions': {
        'IsolationForest': iforest_anomaly_flags,
        'LOF': lof_anomaly_flags,
        'Mahalanobis': mahal_anomaly_flags,
        'PCA': pca_anomaly_flags,
        'DBSCAN': dbscan_anomaly_flags,
    },
    'method_scores': {
        'IsolationForest': iforest_scores,
        'LOF': lof_scores,
        'Mahalanobis': mahal_scores,
        'PCA': pca_scores,
        'DBSCAN': dbscan_scores,
    },

    # Performance metrics
    'performance_summary': results_summary,
    'method_names': method_names,

    # Metadata
    'y_binary': y_binary,
    'y_multilabel': y_multilabel,
    'fault_column_names': fault_column_names,
    'is_manual': is_manual,

    # Configuration
    'config': CONFIG,
}

# Add LSTM results if available
if TF_AVAILABLE:
    phase0_results['method_predictions']['LSTM_AE'] = lstm_anomaly_flags
    phase0_results['method_scores']['LSTM_AE'] = lstm_scores

# Save to pickle
with open(OUTPUT_FILE, 'wb') as f:
    pickle.dump(phase0_results, f)

print(f"✓ Phase 0 results saved to: {OUTPUT_FILE}")
print(f"  File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")

---

## Summary

### Key Findings

**Best Performing Method**: *(Check table above)*

**Ensemble Performance**:
- Detection Rate: *(Check above)*
- False Positive Rate: *(Check above)*
- F1-Score: *(Check above)*

**Insights**:
1. **Method Diversity**: Different methods capture different aspects of precursors
   - Isolation Forest: Global outliers
   - LOF: Local density deviations
   - Mahalanobis: Statistical distance from normal distribution
   - PCA: Deviation from low-dimensional manifold
   - DBSCAN: Clustering-based noise detection
   - LSTM-AE: Temporal pattern anomalies

2. **Ensemble Robustness**: Majority voting reduces false positives while maintaining detection rate

3. **Precursor Signatures**: Events with detected precursors likely exhibit:
   - High CUSUM values (change points)
   - Unusual temporal trends (slopes, accelerations)
   - Deviations in late pre-trigger window (T-34ms to T=0)

### Next Steps

**Phase 1: Binary Classification**
- Use supervised learning to validate fault labels
- Incorporate precursor detection results as features
- Compare against ground truth fault labels

**Phase 2: Multi-Label Triggers**
- Classify which of the 7 ALM fault types are active
- Use precursor patterns to predict fault types

**References**

1. **Isolation Forest**: Liu et al. (2008), "Isolation Forest", IEEE ICDM
2. **LOF**: Breunig et al. (2000), "LOF: Identifying Density-Based Local Outliers", ACM SIGMOD
3. **Mahalanobis Distance**: De Maesschalck et al. (2000), "The Mahalanobis distance", Chemometrics
4. **PCA Anomaly Detection**: Shyu et al. (2003), "A Novel Anomaly Detection Scheme Based on PCA", IEEE
5. **DBSCAN**: Ester et al. (1996), "A Density-Based Algorithm for Discovering Clusters", KDD
6. **LSTM Autoencoder**: Malhotra et al. (2016), "LSTM-based Encoder-Decoder for Multi-sensor Anomaly Detection", ICML
7. **EuXFEL LLRF**: Papers in `/inputs` folder (thppc072.pdf, NAS paper, etc.)
8. **CEBAF LSTM**: Temporal sequence analysis for LLRF fault prediction

---

**End of Notebook 05: Phase 0 - Precursor Detection**
